In [ ]:
#Checking if the library is installed
import pandas as pd
import numpy as np
import re
from datetime import datetime
 #import libary for the gooole play scrapping library
from google_play_scraper import app, reviews, Sort

print("All libraries are installed and imported successfully.")

In [ ]:
from google_play_scraper import app

CBE_APP_ID = "com.combanketh.mobilebanking"

app_info = app(
    CBE_APP_ID,
    lang="en",
    country="et"
)

print("=" * 50)
print("CBE Bank App Metadata")
print("=" * 50)

print(f"App Name: {app_info.get('title')}")
print(f"App Score: {app_info.get('score')}")
print(f"Total Ratings: {app_info.get('ratings')}")
print(f"Installs: {app_info.get('installs')}")
print(f"Reviews Count: {app_info.get('reviews')}")

In [ ]:
# scrape reviews
print("Scraping reviews for CBE Bank App...")
reviews_data, _ = reviews(
    CBE_APP_ID,
    lang="en",
    country="et",
    sort=Sort.NEWEST,
    count=400,  #data amount 400 reviews
    filter_score_with=None
)
print(f"Total reviews scraped: {len(reviews_data)}")

In [ ]:
#single row of the reviews data
print("keys in a single review:")
print(reviews_data[0].keys())

print("Sample review data:")
for key, value in reviews_data[0].items():
    print(f"{key}: {value}")
# Convert the reviews data to a DataFrame

In [ ]:
import pandas as pd

# Extract and clean data
reviews_data_extracted = []
for review in reviews_data:
    content = review.get('content', '')
    # Remove extra spaces
    if content:
        content = ' '.join(content.split())
    
    extracted_review = {
        'reviewId': review.get('reviewId'),
        'content': content,
        'score': review.get('score'),
        'at': review.get('at'),
        'bank': 'CBE Bank'
    }
    reviews_data_extracted.append(extracted_review)

# Create DataFrame
reviews_df = pd.DataFrame(reviews_data_extracted)

# Add Index column starting from 0
reviews_df.insert(0, '', range(len(reviews_df)))

# ====================== DISPLAY ======================
print("\n=== CBE Bank Reviews ===\n")
print(reviews_df.to_string(index=False))

print(f"\nTotal Reviews: {len(reviews_df)}")

In [ ]:
#data values
print(f"Total Reviews: {len(reviews_df)}")
print(f"\nColomns: dtype: ")
print(reviews_df.dtypes)

In [ ]:
print("\nRating Distribution: for CBE Bank App")

ratings_counts = reviews_df['score'].value_counts().sort_index(ascending=False)

for rating, count in ratings_counts.items():
    bar = '█' * (count // 5)
    print(f"Rating {rating}: {count:>4} {bar}")


In [ ]:
#date colomns values check\
reviews_df = pd.DataFrame(reviews_data_extracted)
print("\nDate Column Sample Values:")
print(reviews_df['at'].head(10).to_string())

print("\nData dtypes:")

In [ ]:
print("=" * 50)
print("Data quality check")
print("=" * 50)

#check for missing values
print("Missing values:")
print("-" * 30)
missing_values = reviews_df.isnull().sum()
missing_values_percentage = (missing_values / len(reviews_df) * 100).round(2)
for col in reviews_df.columns:
    status = f"{missing_values[col]} ({missing_values_percentage[col]:.2f}%)"
    print(f"{col}: {status}")



In [ ]:
#duplicate reviews check
print("\nDuplicate reviews check:")
print("-" * 30)

#duplicates based on reviewId
exact_duplicates = reviews_df.duplicated(subset=['reviewId'])
print(f"Exact duplicates based on reviewId: {exact_duplicates.sum()}")

#duplicates on review content(txt)
content_duplicates = reviews_df.duplicated(subset=['content'])
print(f"Duplicates based on review content: {content_duplicates.sum()}")

#empty reviews check
empty_reviews = (reviews_df['content'].str.strip() == '').sum()
print(f"Empty reviews: {empty_reviews}")

In [ ]:
#copying row data
df = reviews_df.copy()
print(f"Starting with: {len(df)} reviews")

In [ ]:
before = len(df)
#drop rows missing critical information 
critical_cols = ['reviewId','score', 'at']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} reviews missing critical information.")
print(f"Remaining reviews after cleaning: {len(df)} reviews")

In [ ]:
#remove duplicates data
df = df.drop_duplicates(subset=['reviewId'], keep='first')

removed = before -len(df)
print(f"Removed {removed} duplicate reviews.")
print(f"Remaining reviews after cleaning: {len(df)} reviews")

In [ ]:
#normalize date format
print("\nNormalizing date format...")
print(df['at'].head(5).to_string())
print(f"dtype : {df['at'].dtype}")

#change the date to YYYY-MM-DD format
df['at'] = pd.to_datetime(df['at']).dt.strftime('%Y-%m-%d')

print("\nDate format normalized. Sample values:")
print(df['at'].head(5).to_string())
print(f"dtype : {df['at'].dtype}")

print(f"\nFinal cleaned dataset contains {len(df)} reviews.")

In [ ]:
#clean text data
def clean_text(text):
    if pd.isnull(text):
        return text
    text = re.sub(r'\s+', ' ', text,) #remove space
    text = text.strip() #remove leading and trailing whitespace
    # Normalize whitespace
    text = ' '.join(text.split())
    return text

sample_text = "   Great    app!\n\nVery useful."
print(f"Before {repr(sample_text)}")
print(f"After {repr(clean_text(sample_text))}")

#apply to full colomns
df['content'] = df['content'].apply(clean_text)

#rmove empty reviews after cleaning
before = len(df)
df = df[df['content'].str.strip() != '']
removed = before - len(df)
print(f"Removed {removed} empty reviews after cleaning.")

In [ ]:
#cheking for invalid ratings
invalid_ratings = df[(df['score'] < 1) | (df['score'] > 5)]
print(f"Invalid ratings found:(outside 1-5) {len(invalid_ratings)}")

#remove invalid ratings
df = df[(df['score']>=1) & (df['score']<=5)]

#ensure score is stored as integer
df['score'] = df['score'].astype(int)

print(f"Remaining: {len(df)} reviews after removing invalid ratings.")
print(f"Rating types: {df['score'].dtype}")

In [ ]:
# Selecting final columns
final_df = df[['reviewId', 'content', 'score', 'at', 'bank']].copy()

# Clean extra spaces in content
final_df['content'] = final_df['content'].astype(str).apply(lambda x: ' '.join(x.split()))

# Sort by date (oldest to newest)
final_df = final_df.sort_values(by='at', ascending=True).reset_index(drop=True)

# Add index column starting from 0
final_df.insert(0, 'index', range(len(final_df)))

# ====================== FINAL DISPLAY ======================
print("\n" + "="*80)
print("                     CBE BANK REVIEWS")
print("="*80)

# Clean and nice tabular display
print(final_df.to_string(index=False, col_space=10))

print("\n" + "="*80)
print(f"Total Reviews: {len(final_df)}")
print("="*80)

In [ ]:
#save to csv
import os
os.makedirs('dataprocessed', exist_ok=True)

clear_output_path = 'dataprocessed/clear_output_cbe_reviews.csv'
final_df.to_csv(clear_output_path, index=False)

print(f"\nCleaned data saved to: {clear_output_path}")
